#Making Community Data FAIR: Steam Tag Taxonomy Pipeline
**Kelsey Kiantoro · INFO628 Data Librarianship & Management · Prof. Vicky Steeves**

### How to use this notebook
Run each cell **one at a time**, top to bottom. Each cell does **one task only**.

| Step | Task |
|------|------|
| 1 | Install packages |
| 2 | Upload & load your CSV |
| 3 | Explore: shape, columns, sample rows |
| 4 | Extract all unique tags |
| 5 | Chart: top 30 most-used tags |
| 6 | Chart: Primary Genre breakdown |
| 7 | Classify tags into dimensions (Grelier framework) |
| 8 | Chart: dimension breakdown |
| 9 | Draw: ERD — the 7-table relational schema |
| 10 | Draw: Network graph — genre hierarchy |
| **10b** | **Metric: Degree centrality — which genres are most connected?** |
| **10c** | **Metric: Tag co-occurrence — which tags appear together most?** |
| 11 | Draw: Sunburst — genre hierarchy interactive |
| 12 | Draw: Swimlane — the full curation workflow |

**Attribution note:** Visualizations and code scaffolding were assisted by Claude (Anthropic, 2025). All analytical decisions, classification rules, taxonomy design, and interpretations are the author's own.

---
## Step 1 — Install packages
Run this first. It installs everything the notebook needs.

In [ ]:
!pip install -q plotly networkx pandas matplotlib
print("✅ Done — packages ready")

---
## Step 2 — Upload and load your CSV
**Upload `steam_games_2026.csv`** using the Files panel on the left (folder icon → upload).
Then run this cell.

In [ ]:
import pandas as pd

# ── Load the CSV
df = pd.read_csv('steam_games_2026.csv', encoding='utf-8-sig')

print(f"Rows    : {len(df)}")
print(f"Columns : {list(df.columns)}")

---
## Step 3 — Explore the data
See what the data looks like.

In [ ]:
# Show first 5 rows
df.head(5)

In [ ]:
# Check column types and missing values
df.info()

In [ ]:
# Basic stats for numeric columns
df[['Price_USD','Review_Score_Pct','Total_Reviews','Estimated_Owners','24h_Peak_Players']].describe().round(2)

---
## Step 4 — Extract all unique tags
Steam stores tags as one cell with semicolons: `'RPG;Action;Co-op'`
This step splits them so we can count each tag individually.

In [ ]:
from collections import Counter

# Split every tag cell by semicolon, flatten into one big list
all_tags_flat = []
for cell in df['All_Tags']:
    for tag in str(cell).split(';'):
        all_tags_flat.append(tag.strip())

# Count how often each tag appears
tag_counts = Counter(all_tags_flat)

print(f"Total tag mentions : {len(all_tags_flat):,}")
print(f"Unique tags        : {len(tag_counts):,}")
print(f"\nTop 10 most used tags:")
for tag, count in tag_counts.most_common(10):
    print(f"  {tag:<30} {count:>5} games")

---
## Step 5 — Chart: Top 30 most-used tags
This shows which tags players use most often across the top 1,000 Steam games.

In [ ]:
import matplotlib.pyplot as plt

# Get top 30
top30 = tag_counts.most_common(30)
tags_labels = [t[0] for t in top30]
tags_values = [t[1] for t in top30]

fig, ax = plt.subplots(figsize=(12, 7), facecolor='#F8F9FA')
ax.set_facecolor('#F8F9FA')

bars = ax.barh(tags_labels[::-1], tags_values[::-1], color='#2E75B6', edgecolor='white', height=0.7)

# Add value labels
for bar, val in zip(bars, tags_values[::-1]):
    ax.text(bar.get_width() + 3, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=8, color='#1F4E79')

ax.set_xlabel('Number of games tagged', fontsize=10)
ax.set_title('Top 30 Steam Tags\n(across 1,000 top games — all tag types mixed)', fontsize=12, fontweight='bold', color='#1F4E79')
ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout()
plt.savefig('top30_tags.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: top30_tags.png")

---
## Step 6 — Chart: Primary Genre breakdown
Steam assigns one main genre per game. This shows how the 1,000 games are distributed.

In [ ]:
# Count games per primary genre
genre_counts = df['Primary_Genre'].value_counts()

fig, ax = plt.subplots(figsize=(9, 5), facecolor='#F8F9FA')
ax.set_facecolor('#F8F9FA')

colors = ['#1F4E79','#2E75B6','#4BACC6','#A9D0E8','#F4A261',
          '#E76F51','#2A9D8F','#264653','#E9C46A','#95a5a6','#6C757D','#adb5bd']

bars = ax.bar(genre_counts.index, genre_counts.values,
              color=colors[:len(genre_counts)], edgecolor='white', linewidth=1.5)

for bar, val in zip(bars, genre_counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 4,
            str(val), ha='center', fontsize=9, fontweight='bold', color='#1F4E79')

ax.set_ylabel('Number of games', fontsize=10)
ax.set_title('Primary Genre Distribution\n(1,000 top Steam games 2024–2026)', fontsize=12, fontweight='bold', color='#1F4E79')
ax.spines[['top','right','left']].set_visible(False)
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.savefig('genre_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: genre_distribution.png")

---
## Step 7 — Classify tags into dimensions
Steam mixes everything into one flat list. `RPG` (genre), `Cyberpunk` (theme), `Co-op` (feature), and `Pixel Graphics` (visual) are treated equally — but they mean completely different things.

This step applies **Grelier, Kaufmann & Schmalz (2023)** to separate them into dimensions.

In [ ]:
# ── Dimension classification map
# Based on Grelier et al. (2023) framework
THEME_TAGS = {
    'Cyberpunk','Post-apocalyptic','Steampunk','Zombies','Anime','Gothic',
    'Mythology','Space','Fantasy','Sci-fi','Western','Medieval','Military',
    'Pirate','Lovecraftian','Ninja','Samurai','Viking','Cats','Dogs',
    'Dinosaurs','Dragons','Demons','Aliens','Robots','Vampire',
    'Supernatural','Psychological','Dark Fantasy','Dystopian','Historical',
    'Cold War','World War II','1980s','Retro','Futuristic','Underwater',
    'Space Opera','Mythology','Noir','Political','Heist','Mafia','Crime',
    'Survival Horror','Lovecraftian','Magic','Martial Arts'
}
FEATURE_TAGS = {
    'Multiplayer','Co-op','PvP','PvE','Online','Local Multiplayer',
    'Singleplayer','Modding','Procedural Generation','Open World','Sandbox',
    'Free to Play','Early Access','Online Co-Op','Cross-Platform Multiplayer',
    'Split Screen','Steam Workshop','Steam Trading Cards','Achievements',
    'Controller','VR Supported','Full controller support','Online PvP',
    'LAN','Asynchronous Multiplayer','Massively Multiplayer','Shared/Split Screen',
    'Steam Cloud','Remote Play Together','Game Controller'
}
VISUAL_TAGS = {
    'Pixel Graphics','Hand-drawn','2D','3D','Anime-style','Colorful',
    'Minimalist','Stylized','Cartoon','Realistic','First-Person','Third Person',
    'Isometric','Top-Down','Side Scroller','2.5D','Low Poly',
    'Voxel','Photorealistic','Cel-shaded'
}
MOOD_TAGS = {
    'Relaxing','Cozy','Atmospheric','Dark','Wholesome','Funny','Cute',
    'Chill','Lighthearted','Emotional','Philosophical','Creepy','Disturbing',
    'Satirical','Humorous','Addictive'
}
ASSESSMENT_TAGS = {
    'Difficult','Short','Long','Choices Matter','Great Soundtrack',
    'Story Rich','Lore-Rich','Masterpiece','Narration','Commentary'
}

# ── Classify each tag
def classify_tag(tag):
    if tag in THEME_TAGS:      return 'Theme'
    if tag in FEATURE_TAGS:    return 'Feature'
    if tag in VISUAL_TAGS:     return 'Visual'
    if tag in MOOD_TAGS:       return 'Tone / Mood'
    if tag in ASSESSMENT_TAGS: return 'Assessment'
    return 'Genre'  # default: anything not categorized above is treated as Genre

# ── Apply to all tags
tag_df = pd.DataFrame(tag_counts.most_common(), columns=['tag_name', 'count'])
tag_df['dimension'] = tag_df['tag_name'].apply(classify_tag)

# ── Summary
summary = tag_df['dimension'].value_counts()
print("=== Tag Dimension Classification ===")
for dim, count in summary.items():
    print(f"  {dim:<15}: {count:>4} tags")
print(f"\n  Total unique tags: {len(tag_df)}")

# Preview
tag_df.head(20)

---
## Step 8 — Chart: Dimension breakdown
Shows how many tags belong to each dimension. Key insight: most of Steam's 'tags' are NOT genres.

In [ ]:
summary = tag_df['dimension'].value_counts()

PALETTE = ['#1F4E79','#2E75B6','#4BACC6','#F4A261','#E76F51','#2A9D8F']

fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor='#F8F9FA')

# ── Left: bar chart by dimension
ax = axes[0]
ax.set_facecolor('#F8F9FA')
bars = ax.barh(summary.index, summary.values, color=PALETTE[:len(summary)], edgecolor='white', height=0.6)
for bar, val in zip(bars, summary.values):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, color='#1F4E79', fontweight='bold')
ax.set_xlabel('Number of unique tags')
ax.set_title('Tags by Dimension\n(Grelier et al. 2023)', fontsize=12, fontweight='bold', color='#1F4E79')
ax.spines[['top','right','left']].set_visible(False)

# ── Right: Genre vs Non-Genre stacked
ax2 = axes[1]
ax2.set_facecolor('#F8F9FA')
genre_n    = int(summary.get('Genre', 0))
non_genre  = int(len(tag_df) - genre_n)
ax2.bar(['All Steam Tags'], [genre_n],    color='#1F4E79', label=f'Genre ({genre_n})',    width=0.4)
ax2.bar(['All Steam Tags'], [non_genre],  bottom=[genre_n], color='#4BACC6',
        label=f'Non-Genre — kept & classified ({non_genre})', width=0.4)
ax2.set_title('Genre vs Non-Genre\n(non-genre = classified, NOT discarded)', fontsize=12, fontweight='bold', color='#1F4E79')
ax2.set_ylabel('Unique tag count')
ax2.legend(fontsize=9)
ax2.spines[['top','right','left']].set_visible(False)
ax2.text(0, genre_n + non_genre + 3,
         f'Total: {len(tag_df)} unique tags', ha='center', fontsize=9, color='#595959', fontstyle='italic')

plt.suptitle('Module 2: Tag Classification Results', fontsize=13, fontweight='bold', color='#1F4E79', y=1.02)
plt.tight_layout()
plt.savefig('tag_dimensions.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: tag_dimensions.png")

---
## Step 9 — ERD: The 7-table relational schema
This shows how the flat CSV was redesigned into a structured relational database.
Each box = one table. Lines = foreign key relationships.

In [ ]:
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch

fig, ax = plt.subplots(figsize=(16, 9), facecolor='#F0F4F8')
ax.set_xlim(0, 16)
ax.set_ylim(0, 9)
ax.axis('off')
ax.set_facecolor('#F0F4F8')

C_ENT  = '#1F4E79'   # entity tables
C_JUN  = '#2E75B6'   # junction tables
C_TEXT = '#FFFFFF'

def draw_table(ax, x, y, w, h, title, fields, color, note=''):
    # Header box
    ax.add_patch(FancyBboxPatch((x, y+h*0.6), w, h*0.4,
        boxstyle='round,pad=0.02', facecolor=color, edgecolor='white', linewidth=2, zorder=3))
    ax.text(x+w/2, y+h*0.81, title, ha='center', va='center',
            fontsize=9, fontweight='bold', color='white', fontfamily='monospace', zorder=4)
    if note:
        ax.text(x+w/2, y+h*0.65, note, ha='center', va='center',
                fontsize=6.5, color='#D6E4F0', fontstyle='italic', zorder=4)
    # Body box
    ax.add_patch(FancyBboxPatch((x, y), w, h*0.6,
        boxstyle='round,pad=0.02', facecolor='white', edgecolor=color, linewidth=1.5, zorder=3))
    row_h = (h * 0.55) / max(len(fields), 1)
    for i, (fname, ftype) in enumerate(fields):
        fy = y + h*0.57 - (i+0.5)*row_h
        icon = '🔑 ' if ftype=='PK' else ('🔗 ' if ftype=='FK' else '   ')
        fc   = '#E76F51' if ftype=='PK' else ('#F4A261' if ftype=='FK' else '#333')
        ax.text(x+0.1, fy, icon+fname, ha='left', va='center',
                fontsize=7.5, color=fc, fontfamily='monospace',
                fontweight='bold' if ftype in ('PK','FK') else 'normal', zorder=4)

def arrow(ax, x1, y1, x2, y2, label=''):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1), zorder=5,
        arrowprops=dict(arrowstyle='->', color='#595959', lw=1.5,
                        connectionstyle='arc3,rad=0.0'))
    if label:
        ax.text((x1+x2)/2, (y1+y2)/2+0.15, label,
                ha='center', fontsize=6.5, color='#595959', fontstyle='italic')

# ── Draw tables
draw_table(ax, 0.2, 5.5, 2.8, 3.0, 'PUBLISHER',
    [('pub_id','PK'),('name',''),('country',''),('size',''),('revenue','')], C_ENT)

draw_table(ax, 4.0, 4.5, 3.0, 4.0, 'GAME',
    [('game_id','PK'),('name',''),('release_date',''),('price_usd',''),
     ('review_score',''),('total_reviews',''),('pub_id','FK')], C_ENT)

draw_table(ax, 8.5, 5.8, 2.8, 2.5, 'TAG',
    [('tag_id','PK'),('tag_name',''),('tag_type_id','FK')], C_ENT)

draw_table(ax, 12.0, 6.5, 3.5, 1.8, 'TAG_TYPE',
    [('type_id','PK'),('dimension','')], C_ENT)

draw_table(ax, 8.5, 2.0, 2.8, 3.2, 'GENRE',
    [('genre_id','PK'),('genre_name',''),('super_genre',''),('parent_id','FK')], C_ENT)

draw_table(ax, 4.0, 0.3, 3.0, 3.2, 'GAME_TAG',
    [('game_id','FK'),('tag_id','FK')], C_JUN, note='junction')

draw_table(ax, 8.5, 0.2, 2.8, 1.5, 'GAME_GENRE',
    [('game_id','FK'),('genre_id','FK')], C_JUN, note='junction')

# ── Arrows
arrow(ax, 3.0, 7.0, 4.0, 7.0, 'publishes')
arrow(ax, 7.0, 7.0, 8.5, 7.0, '')
arrow(ax, 7.0, 5.5, 8.5, 4.0, '')
arrow(ax, 11.3, 7.0, 12.0, 7.3, 'classifies')
arrow(ax, 7.0, 2.0, 8.5, 3.2, '')
arrow(ax, 5.5, 4.5, 5.5, 3.5, '')
arrow(ax, 9.9, 2.0, 9.9, 1.7, '')

# ── Legend
legend_els = [
    mpatches.Patch(color=C_ENT, label='Entity table'),
    mpatches.Patch(color=C_JUN, label='Junction table (many-to-many)'),
    mpatches.Patch(color='#E76F51', label='Primary Key (PK)'),
    mpatches.Patch(color='#F4A261', label='Foreign Key (FK)'),
]
ax.legend(handles=legend_els, loc='lower left', fontsize=8.5, framealpha=0.95)

ax.set_title('ERD — Relational Schema (Logical Level)\nTransformed from flat CSV into 7 normalized tables',
    fontsize=12, fontweight='bold', color='#1F4E79', pad=10)

plt.tight_layout()
plt.savefig('erd_schema.png', dpi=150, bbox_inches='tight', facecolor='#F0F4F8')
plt.show()
print("Saved: erd_schema.png")

---
## Step 10 — Network graph: Genre hierarchy
Shows how genres connect to super-genres. Node size = how many games carry that tag. Larger = more popular.

In [ ]:
import networkx as nx

# ── Build the graph from your actual tag counts
# Genre → Super-genre relationships
hierarchy = [
    # (child_genre, parent_super_genre)
    ('Shooter',          'ACTION'),
    ('FPS',              'ACTION'),
    ('Soulslike',        'ACTION'),
    ('Hack and Slash',   'ACTION'),
    ('Beat em up',       'ACTION'),
    ('Bullet Hell',      'ACTION'),
    ('Roguelike',        'ACTION'),
    ('Roguelite',        'ACTION'),
    ('JRPG',             'RPG'),
    ('Action RPG',       'RPG'),
    ('Dungeon Crawler',  'RPG'),
    ('Turn-Based',       'STRATEGY'),
    ('Tower Defense',    'STRATEGY'),
    ('City Builder',     'STRATEGY'),
    ('Grand Strategy',   'STRATEGY'),
    ('Metroidvania',     'ADVENTURE'),
    ('Visual Novel',     'ADVENTURE'),
    ('Point & Click',    'ADVENTURE'),
    ('Walking Simulator','ADVENTURE'),
    ('Farming Sim',      'SIMULATION'),
    ('Life Sim',         'SIMULATION'),
    ('City Builder',     'SIMULATION'),
    ('Puzzle',           'CASUAL'),
    ('Card Game',        'CASUAL'),
    ('Idle',             'CASUAL'),
]

# Super-genres and their colors
SUPER_COLORS = {
    'ACTION':'#E76F51','RPG':'#2E75B6','STRATEGY':'#264653',
    'ADVENTURE':'#2A9D8F','SIMULATION':'#F4A261','CASUAL':'#4BACC6',
}

G = nx.DiGraph()

# Add super-genre nodes
for sg, col in SUPER_COLORS.items():
    count = tag_counts.get(sg.capitalize(), tag_counts.get(sg, 5000))
    G.add_node(sg, level='super', color=col, size=count)

# Add genre nodes and edges
for genre, super_g in hierarchy:
    count = tag_counts.get(genre, 500)
    color = SUPER_COLORS.get(super_g, '#A9D0E8')
    G.add_node(genre, level='genre', color=color+'88', size=count)
    G.add_edge(super_g, genre)

# Layout
pos = nx.spring_layout(G, k=2.8, seed=42)

super_nodes = [n for n,d in G.nodes(data=True) if d.get('level')=='super']
genre_nodes = [n for n,d in G.nodes(data=True) if d.get('level')=='genre']

fig, ax = plt.subplots(figsize=(15, 9), facecolor='#0D1B2A')
ax.set_facecolor('#0D1B2A')

# Draw edges
nx.draw_networkx_edges(G, pos, ax=ax, edge_color='#4BACC6',
    alpha=0.35, arrows=True, arrowsize=12, width=1.2)

# Draw super-genre nodes (bigger)
nx.draw_networkx_nodes(G, pos, nodelist=super_nodes, ax=ax,
    node_size=[G.nodes[n]['size']/5 for n in super_nodes],
    node_color=[G.nodes[n]['color'] for n in super_nodes], alpha=0.95)

# Draw genre nodes (smaller)
nx.draw_networkx_nodes(G, pos, nodelist=genre_nodes, ax=ax,
    node_size=[G.nodes[n]['size']/8 for n in genre_nodes],
    node_color=[G.nodes[n]['color'] for n in genre_nodes], alpha=0.75)

# Labels
nx.draw_networkx_labels(G, pos,
    labels={n:n for n in super_nodes}, ax=ax,
    font_size=9, font_color='white', font_weight='bold')
nx.draw_networkx_labels(G, pos,
    labels={n:n for n in genre_nodes}, ax=ax,
    font_size=7, font_color='#A9D0E8')

ax.set_title('Genre Hierarchy Network\nNode size = tag frequency in dataset  |  Edge = parent → child',
    fontsize=12, color='white', fontweight='bold')
ax.axis('off')
plt.tight_layout()
plt.savefig('network_graph.png', dpi=150, bbox_inches='tight', facecolor='#0D1B2A')
plt.show()
print("Saved: network_graph.png")

---
## Step 10b — Network metrics: Degree centrality

**What is degree centrality?**
For each genre node, it measures: *how many other nodes connect to it, as a fraction of all possible connections.*

A score of `1.0` means connected to everything. A score of `0.05` means connected to very few.
This tells you which genres are the most central 'hubs' in the tag ecosystem — without any NLP, just counting connections.

In [ ]:
import networkx as nx

# ── Degree centrality: how connected is each node?
# (G was built in Step 10 above — make sure you ran that cell first)
degree_centrality = nx.degree_centrality(G)

# Sort highest to lowest
ranked = sorted(degree_centrality.items(), key=lambda x: x[1], reverse=True)

print("=== Degree Centrality: Most Connected Genre Nodes ===")
print(f"{'Genre':<25} {'Score':>8}  Meaning")
print("-" * 65)
for node, score in ranked:
    level = G.nodes[node].get('level', 'genre')
    tag_freq = G.nodes[node].get('size', 0)
    print(f"{node:<25} {score:>8.3f}  [{level}]  {tag_freq:,} games tagged")

print()
print("── How to read this ──────────────────────────────────────")
print("High score = that genre connects to many others (a hub)")
print("Low score  = that genre is a leaf / niche subgenre")
print("Super-genres (ACTION, RPG, etc.) should score highest")
print("because they parent many child genres.")

---
## Step 10c — Tag co-occurrence (no NLP needed)

**What is co-occurrence?**
Two tags *co-occur* when they appear on the same game.
If `Roguelike` and `Card Game` both appear on 47 games, their co-occurrence count is 47.

This is calculated with `itertools.combinations` — no NLTK, no spaCy, just Python counting pairs.

**What this tells you:** Which genre combinations are most common in the top 1,000 Steam games.

In [ ]:
from itertools import combinations
from collections import Counter

# ── Count how often tag pairs appear on the same game
# This is co-occurrence: no NLP, just pair counting
co_pairs = Counter()

for tags_str in df['All_Tags']:
    game_tags = [t.strip() for t in str(tags_str).split(';')]
    # Get all unique pairs from this game's tag list
    for tag_a, tag_b in combinations(sorted(set(game_tags)), 2):
        co_pairs[(tag_a, tag_b)] += 1

print(f"Total unique tag pairs found: {len(co_pairs):,}")
print()
print("=== Top 20 Most Co-occurring Tag Pairs ===")
print(f"{'Tag A':<25} {'Tag B':<25} {'Games together':>15}")
print("-" * 68)
for (tag_a, tag_b), count in co_pairs.most_common(20):
    print(f"{tag_a:<25} {tag_b:<25} {count:>15,}")

print()
print("── How to explain this to your professor ─────────────────")
print("'I calculated tag co-occurrence using itertools.combinations.")
print("For each game, I took all tag pairs and counted how often")
print("each pair appeared together across 1,000 games.'")
print("No NLP. No adjacency matrix. Just pair counting.")

In [ ]:
# ── Visualize top co-occurring pairs as a bar chart
import matplotlib.pyplot as plt

top_pairs = co_pairs.most_common(15)
pair_labels = [f"{a} + {b}" for (a,b),_ in top_pairs]
pair_values = [c for _,c in top_pairs]

fig, ax = plt.subplots(figsize=(12, 6), facecolor='#F8F9FA')
ax.set_facecolor('#F8F9FA')

bars = ax.barh(pair_labels[::-1], pair_values[::-1],
               color='#2E75B6', edgecolor='white', height=0.65)
for bar, val in zip(bars, pair_values[::-1]):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=8.5, color='#1F4E79')

ax.set_xlabel('Number of games where both tags appear together', fontsize=10)
ax.set_title('Top 15 Tag Co-occurrence Pairs\n'
             '(how often two tags appear on the same game — no NLP, just counting)',
             fontsize=12, fontweight='bold', color='#1F4E79')
ax.spines[['top','right','left']].set_visible(False)
plt.tight_layout()
plt.savefig('cooccurrence_pairs.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: cooccurrence_pairs.png")

---
## Step 11 — Sunburst chart: Genre hierarchy (interactive)
Click on any slice to zoom in. Hover to see tag counts. Built from your actual data.

In [ ]:
import plotly.graph_objects as go

# ── Build sunburst rows from real tag counts
def tag_count(name):
    """Get count from actual dataset, fall back to 100"""
    return tag_counts.get(name, 100)

rows = [
    # Root
    {'id':'Root',        'label':'Steam\nTags',   'parent':'',     'value':0,                  'color':'#1F4E79'},
    # Super-genres
    {'id':'ACTION',      'label':'Action',         'parent':'Root', 'value':tag_count('Action'),'color':'#E76F51'},
    {'id':'RPG',         'label':'RPG',            'parent':'Root', 'value':tag_count('RPG'),   'color':'#2E75B6'},
    {'id':'ADVENTURE',   'label':'Adventure',      'parent':'Root', 'value':tag_count('Adventure'),'color':'#2A9D8F'},
    {'id':'SIMULATION',  'label':'Simulation',     'parent':'Root', 'value':tag_count('Simulation'),'color':'#F4A261'},
    {'id':'STRATEGY',    'label':'Strategy',       'parent':'Root', 'value':tag_count('Strategy'),'color':'#264653'},
    {'id':'CASUAL',      'label':'Casual',         'parent':'Root', 'value':tag_count('Casual'), 'color':'#4BACC6'},
    {'id':'INDIE',       'label':'Indie',          'parent':'Root', 'value':tag_count('Indie'),  'color':'#6C757D'},
    # ACTION → genres
    {'id':'Roguelike',   'label':'Roguelike',      'parent':'ACTION','value':tag_count('Roguelike'),'color':'#F4A261'},
    {'id':'Roguelite',   'label':'Roguelite',      'parent':'ACTION','value':tag_count('Roguelite'),'color':'#F4A261'},
    {'id':'Shooter',     'label':'Shooter',        'parent':'ACTION','value':tag_count('Shooter'), 'color':'#F4A261'},
    {'id':'Soulslike',   'label':'Soulslike',      'parent':'ACTION','value':tag_count('Soulslike'),'color':'#F4A261'},
    {'id':'Platformer',  'label':'Platformer',     'parent':'ACTION','value':tag_count('Platformer'),'color':'#F4A261'},
    # Shooter → sub-genres
    {'id':'FPS',         'label':'FPS',            'parent':'Shooter','value':tag_count('FPS'),   'color':'#E9C46A'},
    {'id':'BH',          'label':'Bullet Hell',    'parent':'Shooter','value':tag_count('Bullet Hell'),'color':'#E9C46A'},
    {'id':'ExtShooter',  'label':'Extraction Shooter','parent':'Shooter','value':tag_count('Extraction Shooter'),'color':'#E9C46A'},
    # RPG → genres
    {'id':'JRPG',        'label':'JRPG',           'parent':'RPG',  'value':tag_count('JRPG'),    'color':'#4BACC6'},
    {'id':'ActionRPG',   'label':'Action RPG',     'parent':'RPG',  'value':tag_count('Action RPG'),'color':'#4BACC6'},
    {'id':'DC',          'label':'Dungeon Crawler', 'parent':'RPG',  'value':tag_count('Dungeon Crawler'),'color':'#4BACC6'},
    # ADVENTURE → genres
    {'id':'MV',          'label':'Metroidvania',   'parent':'ADVENTURE','value':tag_count('Metroidvania'),'color':'#A9D0E8'},
    {'id':'VN',          'label':'Visual Novel',   'parent':'ADVENTURE','value':tag_count('Visual Novel'),'color':'#A9D0E8'},
    # SIMULATION → genres
    {'id':'FarmSim',     'label':'Farming Sim',    'parent':'SIMULATION','value':tag_count('Farming Sim'),'color':'#2A9D8F'},
    {'id':'CB',          'label':'City Builder',   'parent':'SIMULATION','value':tag_count('City Builder'),'color':'#2A9D8F'},
    # STRATEGY → genres
    {'id':'TBS',         'label':'Turn-Based',     'parent':'STRATEGY','value':tag_count('Turn-Based'),'color':'#95a5a6'},
    {'id':'TD',          'label':'Tower Defense',  'parent':'STRATEGY','value':tag_count('Tower Defense'),'color':'#95a5a6'},
]

import pandas as pd
df_sun = pd.DataFrame(rows)

fig = go.Figure(go.Sunburst(
    ids=df_sun['id'],
    labels=df_sun['label'],
    parents=df_sun['parent'],
    values=df_sun['value'],
    marker=dict(colors=df_sun['color']),
    branchvalues='remainder',
    hovertemplate='<b>%{label}</b><br>Games tagged: %{value:,}<extra></extra>',
    maxdepth=3,
    insidetextfont=dict(size=11),
))

fig.update_layout(
    title=dict(
        text='<b>Steam Genre Taxonomy — Sunburst</b><br>'
             '<sup>Click a slice to zoom in | Size = games tagged in your 1,000-game dataset</sup>',
        font=dict(size=15, color='#1F4E79'), x=0.5
    ),
    paper_bgcolor='#F8F9FA',
    width=700, height=650,
    margin=dict(t=90, l=10, r=10, b=10)
)
fig.show()
fig.write_html('sunburst_genre.html')
print("Saved: sunburst_genre.html")

---
## Step 12 — Swimlane: The curation workflow
Shows the full journey from raw community data → FAIR-compliant dataset.
Each horizontal lane = one layer of the pipeline.

In [ ]:
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(18, 8), facecolor='#F8F9FA')
ax.set_xlim(0, 18)
ax.set_ylim(0, 8)
ax.axis('off')

LANES = [
    ('Community Data',       '#1F4E79'),
    ('Raw Input (CSV)',      '#2E75B6'),
    ('Curation & Cleaning',  '#2A9D8F'),
    ('Relational Structure', '#F4A261'),
    ('FAIR Output',          '#E76F51'),
]
lane_h = 8.0 / len(LANES)

# ── Draw lane backgrounds
for i, (label, color) in enumerate(LANES):
    y = 8 - (i+1)*lane_h
    ax.add_patch(plt.Rectangle((0, y), 18, lane_h, facecolor=color, alpha=0.07, zorder=0))
    ax.add_patch(plt.Rectangle((0, y), 2.4, lane_h, facecolor=color, alpha=0.88, zorder=1))
    ax.text(1.2, y + lane_h/2, label, ha='center', va='center',
            fontsize=9, color='white', fontweight='bold', zorder=2)
    ax.axhline(y, color='white', linewidth=1.5, zorder=1)

# ── Process boxes: (x_center, lane_index, title, subtitle)
BOXES = [
    (4.5,  0, 'Steam Tags\n(folksonomy)',    '370 unique tags\nvote-weighted, no hierarchy'),
    (10.5, 0, 'MobyGames\n(taxonomy)',       '230 controlled terms\neditor-assigned'),
    (15.5, 0, 'Gap Analysis',                '54 shared  |  316 Steam-only\n176 MobyGames-only'),
    (4.5,  1, 'Steam 2026\nCSV',             '1,000 games\n12 columns'),
    (10.5, 1, 'Gamalytic\nPublisher Data',   '206 publishers\nsize + revenue tier'),
    (4.5,  2, 'OpenRefine\nCleaning',        'Normalize names\nFlag missing values'),
    (10.5, 2, 'Python\nClassification',      'Regex + manual review\n370 tags → dimensions'),
    (15.5, 2, 'Sentence\nTransformer',       'HuggingFace mapping\nSteam ↔ MobyGames'),
    (6.5,  3, 'SQLite\nDatabase',            '7 tables  |  3-level ERD\nFK constraints enforced'),
    (13.0, 3, 'Genre Taxonomy',              '9 super-genres\n78 genres  |  41 sub-genres'),
    (5.0,  4, 'SKOS Schema',                 'prefLabel = MobyGames\naltLabel = Steam tags'),
    (11.0, 4, 'DataCite\nMetadata',          'Creator + contributors\nODC-By license'),
    (16.0, 4, '✅ FAIR\nDataset',            'Findable · Accessible\nInteroperable · Reusable'),
]

BW, BH = 2.5, lane_h * 0.70

for (xc, li, title, sub) in BOXES:
    y_lane = 8 - (li+1)*lane_h
    yc = y_lane + lane_h/2
    color = LANES[li][1]
    ax.add_patch(FancyBboxPatch((xc-BW/2, yc-BH/2), BW, BH,
        boxstyle='round,pad=0.06', facecolor='white', edgecolor=color, linewidth=2, zorder=3))
    ax.text(xc, yc+BH*0.14, title, ha='center', va='center',
            fontsize=8.5, fontweight='bold', color=color, zorder=4)
    ax.text(xc, yc-BH*0.28, sub, ha='center', va='center',
            fontsize=6.8, color='#595959', zorder=4)

# ── Arrows (vertical: raw → curation → structure → output)
def arr(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2,y2), xytext=(x1,y1), zorder=5,
        arrowprops=dict(arrowstyle='->', color='#595959', lw=1.5,
                        connectionstyle='arc3,rad=0'))

arr(ax, 4.5,  8-1.5*lane_h, 4.5,  8-2.0*lane_h)   # CSV → cleaning
arr(ax, 10.5, 8-1.5*lane_h, 10.5, 8-2.0*lane_h)   # Publisher → cleaning
arr(ax, 6.5,  8-2.5*lane_h, 6.5,  8-3.0*lane_h)   # cleaning → DB
arr(ax, 10.5, 8-2.5*lane_h, 13.0, 8-3.0*lane_h)   # classification → taxonomy
arr(ax, 6.5,  8-3.5*lane_h, 5.0,  8-4.0*lane_h)   # DB → SKOS
arr(ax, 13.0, 8-3.5*lane_h, 11.0, 8-4.0*lane_h)   # taxonomy → DataCite
arr(ax, 7.2,  8-4.5*lane_h, 9.8,  8-4.5*lane_h)   # SKOS → DataCite
arr(ax, 12.2, 8-4.5*lane_h, 14.8, 8-4.5*lane_h)   # DataCite → FAIR
arr(ax, 15.5, 8-0.5*lane_h, 15.5, 8-2.5*lane_h)   # Gap → Sentence Transformer

ax.set_title('Swimlane: From Raw Community Data → FAIR-Compliant Dataset',
    fontsize=13, fontweight='bold', color='#1F4E79', pad=10, x=0.5)

plt.tight_layout()
plt.savefig('swimlane.png', dpi=150, bbox_inches='tight', facecolor='#F8F9FA')
plt.show()
print("Saved: swimlane.png")

---
## ✅ What you just built

| File saved | What it shows |
|------------|---------------|
| `top30_tags.png` | Most-used Steam tags across 1,000 games |
| `genre_distribution.png` | Primary genre breakdown |
| `tag_dimensions.png` | Tags sorted by dimension (Grelier framework) |
| `erd_schema.png` | 7-table relational schema |
| `network_graph.png` | Genre hierarchy as a network |
| `cooccurrence_pairs.png` | Top tag pairs that appear on the same game |
| `sunburst_genre.html` | Interactive sunburst — click to explore |
| `swimlane.png` | Full curation workflow |

**Core finding visible across all charts:**
Steam treats genre, theme, feature, mood, and visual style as one flat list.
The ERD + taxonomy work makes those hidden dimensions explicit — that *is* what FAIR means in practice.

**Attribution:**
Visualizations and code scaffolding assisted by Claude (Anthropic, 2025).
All classification decisions, taxonomy design, database structure, and analytical interpretations are the author's own.

---
*Kelsey Kiantoro · INFO628 · Prof. Vicky Steeves · May 2026*